# Lesson 16a: Reinforcement Learning from Human Feedback (RLHF) Theory

**Learning Objectives:**
- Understand reward learning from human preferences
- Master the RLHF pipeline (SFT → Reward Model → RL)
- Learn PPO-RLHF for language model alignment
- Study Direct Preference Optimization (DPO)
- Explore Constitutional AI and safety mechanisms
- Understand red teaming and robustness

**Prerequisites:** Policy Gradients (8a), PPO (9a), Multi-Agent (13a)

**References:**
- Christiano et al. (2017) - Deep RL from Human Preferences
- Ouyang et al. (2022) - InstructGPT (ChatGPT)
- Rafailov et al. (2023) - Direct Preference Optimization
- Bai et al. (2022) - Constitutional AI
- Ziegler et al. (2019) - Fine-Tuning Language Models from Human Preferences

## 1. Motivation: Why RLHF?

### The Alignment Problem

**Challenge**: How do we specify what we want AI systems to do?

**Traditional RL issues:**
- Hand-crafted rewards are brittle
- Reward hacking and specification gaming
- "You get what you measure, not what you want"

**RLHF Solution:**
1. Learn reward from human preferences
2. Optimize policy via RL with learned reward
3. Human feedback grounds desired behavior

### Success Stories

- **ChatGPT/GPT-4**: RLHF for helpfulness, honesty, harmlessness
- **Claude**: Constitutional AI with RLHF
- **Robotics**: Learning from human demonstrations and preferences
- **Game playing**: DeepMind's Atari agents from human feedback

## 2. Preference Learning Framework

### Bradley-Terry Model

**Setup:**
- Agent generates trajectories: $\tau_1, \tau_2$
- Human provides preference: $\tau_1 \succ \tau_2$ (prefer $\tau_1$)

**Preference probability:**
$$P(\tau_1 \succ \tau_2) = \frac{\exp(r(\tau_1))}{\exp(r(\tau_1)) + \exp(r(\tau_2))}$$

Equivalently:
$$P(\tau_1 \succ \tau_2) = \sigma(r(\tau_1) - r(\tau_2))$$

where $\sigma$ is the sigmoid function.

### Reward Model Training

**Objective**: Learn reward function $r_\theta$ that predicts preferences

**Loss function:**
$$\mathcal{L}(\theta) = -\mathbb{E}_{(\tau_1 \succ \tau_2) \sim \mathcal{D}} \left[ \log \sigma(r_\theta(\tau_1) - r_\theta(\tau_2)) \right]$$

**Dataset $\mathcal{D}$**: Collection of human preference comparisons

### Trajectory Reward

For language models, trajectory = sequence of tokens:
$$r(\tau) = r(x, y) = \sum_{t=1}^{|y|} r_\theta(x, y_{1:t})$$

where:
- $x$ = prompt
- $y$ = response
- Often simplified to: $r(x, y) = r_\theta(x, y)$ (reward at end)

## 3. The RLHF Pipeline

### Three-Stage Process

**Stage 1: Supervised Fine-Tuning (SFT)**
- Collect high-quality demonstrations
- Fine-tune pretrained LM on demonstrations
- Creates initial policy: $\pi^{\text{SFT}}$

**Stage 2: Reward Model Training**
- Sample responses from $\pi^{\text{SFT}}$
- Collect human preference comparisons
- Train reward model $r_\theta$ via Bradley-Terry

**Stage 3: RL Fine-Tuning**
- Optimize policy $\pi_\phi$ using PPO
- Reward: $r_\theta$ from Stage 2
- KL penalty to prevent drift from $\pi^{\text{SFT}}$

### Reward Model Architecture

**For language models:**
1. Start with pretrained LM (e.g., GPT)
2. Remove final token prediction head
3. Add scalar reward head: $r_\theta(x, y) \in \mathbb{R}$
4. Train on preference data

**Key insight**: Reward model shares architecture with policy, learns value judgments

## 4. PPO-RLHF (InstructGPT / ChatGPT)

### Objective Function

**RL objective with KL penalty:**
$$\max_{\pi_\phi} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\phi(y|x)} \left[ r_\theta(x, y) - \beta \log \frac{\pi_\phi(y|x)}{\pi^{\text{SFT}}(y|x)} \right]$$

**Components:**
- $r_\theta(x, y)$: Learned reward from preference model
- $\beta \text{KL}(\pi_\phi \| \pi^{\text{SFT}})$: Regularization to prevent mode collapse
- $\beta$: KL coefficient (typically 0.01 - 0.1)

### Why KL Penalty?

**Without KL penalty:**
- Policy can exploit reward model errors
- May produce nonsensical outputs that fool $r_\theta$
- Catastrophic forgetting of language skills

**With KL penalty:**
- Policy stays close to SFT initialization
- Maintains fluency and coherence
- Bounded exploration

### PPO Implementation for LMs

**Per-token rewards:**
$$r_t(x, y_{1:t}) = \begin{cases}
r_\theta(x, y) - \beta \log \frac{\pi_\phi(y_t|x, y_{<t})}{\pi^{\text{SFT}}(y_t|x, y_{<t})} & t = |y| \\
-\beta \log \frac{\pi_\phi(y_t|x, y_{<t})}{\pi^{\text{SFT}}(y_t|x, y_{<t})} & t < |y|
\end{cases}$$

**Value function**: $V_\psi(x, y_{1:t})$ estimates future rewards

**Advantage**: $A_t = r_t + \gamma V_\psi(x, y_{1:t+1}) - V_\psi(x, y_{1:t})$

**PPO clipped objective:**
$$L^{\text{CLIP}}(\phi) = \mathbb{E}_t \left[ \min(\rho_t A_t, \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_t) \right]$$

where $\rho_t = \frac{\pi_\phi(y_t|x, y_{<t})}{\pi_{\phi_{\text{old}}}(y_t|x, y_{<t})}$

## 5. Direct Preference Optimization (DPO)

### Motivation

**RLHF challenges:**
- Requires separate reward model
- PPO training is unstable
- Computationally expensive (3 models: policy, value, reward)

**DPO insight**: Can optimize policy directly from preferences without explicit reward model!

### Mathematical Derivation

**Start with optimal policy for KL-constrained RL:**
$$\pi^*(y|x) = \frac{1}{Z(x)} \pi^{\text{SFT}}(y|x) \exp\left(\frac{1}{\beta} r(x, y)\right)$$

**Rearrange to get reward:**
$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi^{\text{SFT}}(y|x)} + \beta \log Z(x)$$

**Substitute into Bradley-Terry model:**
$$P(y_1 \succ y_2 | x) = \sigma\left(\beta \log \frac{\pi^*(y_1|x)}{\pi^{\text{SFT}}(y_1|x)} - \beta \log \frac{\pi^*(y_2|x)}{\pi^{\text{SFT}}(y_2|x)}\right)$$

**DPO Loss:**
$$\mathcal{L}_{\text{DPO}}(\pi_\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi^{\text{SFT}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi^{\text{SFT}}(y_l|x)} \right) \right]$$

where:
- $y_w$ = preferred (winning) response
- $y_l$ = dis-preferred (losing) response

### DPO vs RLHF

| Aspect | RLHF | DPO |
|--------|------|-----|
| Reward model | Explicit $r_\theta$ | Implicit via policy ratio |
| Training | PPO (3 models) | Direct supervised learning |
| Stability | Can be unstable | More stable |
| Compute | 3x model memory | 2x model memory |
| Performance | Slightly better | Comparable |

**Key insight**: DPO is RLHF with closed-form optimal policy substitution!

## 6. Constitutional AI (Anthropic)

### Core Principles

**Problem with standard RLHF:**
- Human preferences can be biased, harmful, or inconsistent
- Direct optimization can amplify worst human tendencies
- Scalability: human feedback is expensive

**Constitutional AI solution:**
1. Define explicit principles ("constitution")
2. Use AI feedback to enforce principles
3. Reduce reliance on human feedback

### Two-Phase Training

**Phase 1: Supervised Constitutional Learning**
1. Generate harmful responses from initial model
2. Ask model to critique based on principles
3. Ask model to revise response to be harmless
4. Fine-tune on revised responses

**Phase 2: RL from AI Feedback (RLAIF)**
1. Generate pairs of responses
2. Ask model which better follows principles
3. Train reward model on AI preferences
4. Run PPO with learned reward

### Example Principles

```
1. Choose the response that is most helpful, honest, and harmless.
2. Choose the response that is least likely to encourage illegal or unethical behavior.
3. Choose the response that sounds most like something a wise, ethical person would say.
4. Choose the response that avoids discrimination and bias.
```

### Benefits

- **Transparency**: Explicit values rather than implicit
- **Scalability**: AI feedback cheaper than human
- **Consistency**: Principles applied uniformly
- **Safety**: Reduces harmful outputs

## 7. Data Collection and Labeling

### Prompt Distribution

**Sources:**
- User queries (real or synthetic)
- Red team adversarial prompts
- Diverse topics and difficulty levels

**Key considerations:**
- Representativeness of deployment distribution
- Coverage of edge cases
- Balance of task types

### Response Generation

**For preference comparisons:**
1. Sample K responses from policy: $y_1, \ldots, y_K \sim \pi(\cdot|x)$
2. Present pairs to labelers: $(y_i, y_j)$
3. Collect preference: $y_i \succ y_j$ or $y_j \succ y_i$

**Sampling strategies:**
- **Temperature sampling**: Higher T for diversity
- **Top-k sampling**: Truncate low-probability tokens
- **Best-of-N**: Sample N, use reward model to rank

### Labeling Interface

**Binary comparison:**
```
Which response is better?
[ ] Response A
[ ] Response B
[ ] Tie
```

**Detailed criteria:**
- Helpfulness: Does it answer the question?
- Truthfulness: Is information accurate?
- Harmlessness: Avoids harmful content?

### Data Quality

**Inter-annotator agreement:**
- Cohen's kappa: Measure consistency
- Typical RLHF: κ ≈ 0.6-0.7 (substantial agreement)

**Quality control:**
- Expert labelers with training
- Multiple labels per comparison
- Regular calibration sessions
- Filter low-confidence labels

## 8. Red Teaming and Robustness

### Adversarial Testing

**Red teaming**: Actively try to elicit harmful outputs

**Attack categories:**
1. **Jailbreaks**: Bypass safety guardrails
2. **Prompt injection**: Manipulate system instructions
3. **Toxicity**: Elicit offensive content
4. **Misinformation**: Generate false claims
5. **Privacy**: Leak training data

### Defense Strategies

**During training:**
- Include adversarial examples in preference data
- Reward model trained to recognize attacks
- Constitutional principles against harmful content

**At inference:**
- Input filters: Detect malicious prompts
- Output filters: Block harmful generations
- Moderation API: Real-time safety checks

### Iterative Red Teaming

**Process:**
1. Deploy model to red team
2. Collect adversarial prompts
3. Generate safe responses
4. Add to training data
5. Retrain model
6. Repeat

**Result**: Progressively more robust models

## 9. Evaluation Metrics

### Automatic Metrics

**Reward model score:**
$$\text{RM-Score} = \mathbb{E}_{x \sim \mathcal{D}_{\text{test}}, y \sim \pi(\cdot|x)} [r_\theta(x, y)]$$

**Win rate against baseline:**
$$\text{WinRate} = P(\pi(\cdot|x) \succ \pi_{\text{base}}(\cdot|x))$$

**KL divergence from reference:**
$$\text{KL} = \mathbb{E}_{x, y} \left[ \log \frac{\pi(y|x)}{\pi^{\text{SFT}}(y|x)} \right]$$

### Human Evaluation

**Gold standard**: Live human preferences

**Protocol:**
1. Sample prompts from test set
2. Generate responses from multiple models
3. Blind comparison by human raters
4. Aggregate win rates

**Elo ratings**: Rank models via pairwise comparisons

### Safety Metrics

**Harmlessness:**
- Toxicity rate (via Perspective API)
- Refusal rate on harmful prompts
- Red team success rate

**Truthfulness:**
- TruthfulQA benchmark
- Factual accuracy on knowledge tasks
- Calibration of uncertainty

## 10. Challenges and Open Problems

### Reward Hacking

**Problem**: Policy exploits reward model misspecification

**Example**: Model generates long, verbose answers to maximize reward even when concise answer is better

**Mitigation:**
- Strong KL penalty
- Ensemble of reward models
- Iterative reward model updates

### Scalable Oversight

**Problem**: Humans can't evaluate superhuman AI

**Research directions:**
- **Recursive reward modeling**: Use AI to help evaluate AI
- **Debate**: AI systems argue; human judges winner
- **Amplification**: Break hard tasks into easier subtasks

### Distributional Shift

**Problem**: Deployment distribution differs from training

**Solutions:**
- Continuous learning from deployment
- Active learning for OOD examples
- Robust reward models

### Value Alignment

**Deep question**: Whose values should AI optimize?

**Considerations:**
- Cultural diversity
- Individual preferences vs. societal norms
- Long-term vs. short-term tradeoffs

### Compute Efficiency

**Challenge**: RLHF requires massive compute

**Directions:**
- DPO and other direct methods
- Smaller reward models
- Offline RL to reduce sampling
- Model distillation

## 11. Advanced Topics

### Multi-Objective RLHF

**Goal**: Optimize multiple objectives (helpfulness, harmlessness, etc.)

**Approach 1: Weighted reward**
$$r(x, y) = \alpha_1 r_1(x, y) + \alpha_2 r_2(x, y) + \ldots$$

**Approach 2: Pareto optimization**
- Find policies that are Pareto optimal
- User chooses tradeoff at inference

### Active Learning for Preferences

**Goal**: Query most informative comparisons

**Uncertainty sampling:**
$$x^* = \arg\max_x \mathbb{H}[P(y_1 \succ y_2 | x)]$$

**Disagreement between ensemble:**
- Train ensemble of reward models
- Query where models disagree most

### Reinforcement Learning from AI Feedback (RLAIF)

**Fully automated pipeline:**
1. Use AI to generate preferences (no human labels)
2. Train reward model on AI preferences
3. Run PPO with learned reward

**Promise**: Cheaper and more scalable than human feedback

**Risk**: AI biases amplified without human oversight

### Process-Based Feedback

**Outcome-based**: Judge final answer only

**Process-based**: Judge reasoning steps

**Benefits:**
- Catches flawed reasoning even if answer is correct
- More interpretable
- Better generalization

**Implementation**: Reward each reasoning step, not just final output

## 12. Practical Considerations

### Hyperparameters

**Reward model:**
- Learning rate: 1e-5 to 1e-6
- Batch size: 32-128 comparisons
- Training steps: 1-2 epochs

**PPO-RLHF:**
- KL coefficient β: 0.01-0.1
- PPO clip ε: 0.1-0.2
- Value loss coefficient: 0.5-1.0
- Learning rate: 1e-6 to 1e-5
- Batch size: 64-512 sequences

**DPO:**
- β (temperature): 0.1-0.5
- Learning rate: 1e-6 to 5e-7
- Batch size: 32-128 comparisons

### Implementation Tips

1. **Start with SFT**: Strong initialization is critical
2. **Monitor KL**: Don't let policy drift too far
3. **Validate reward model**: Check on held-out preferences
4. **Use multiple reward models**: Ensemble for robustness
5. **Log everything**: Track reward, KL, response length, etc.
6. **Test early and often**: Human eval throughout training

### Common Pitfalls

❌ **Too high KL penalty**: Policy doesn't improve
✅ **Solution**: Lower β gradually

❌ **Too low KL penalty**: Mode collapse, nonsensical outputs
✅ **Solution**: Increase β, use adaptive KL

❌ **Weak reward model**: Policy doesn't learn meaningful signal
✅ **Solution**: Collect more/better preferences, larger RM

❌ **Distribution mismatch**: Training prompts ≠ deployment
✅ **Solution**: Collect diverse, representative prompts

## Summary

### Key Concepts

1. **RLHF Pipeline**: SFT → Reward Model → RL Fine-tuning
2. **Preference Learning**: Bradley-Terry model for reward from comparisons
3. **PPO-RLHF**: Optimize policy with learned reward + KL penalty
4. **DPO**: Direct preference optimization without explicit reward model
5. **Constitutional AI**: Principles-based training with AI feedback
6. **Safety**: Red teaming, robustness, alignment

### Mathematical Foundations

**Bradley-Terry**: $P(y_1 \succ y_2) = \sigma(r(y_1) - r(y_2))$

**RLHF Objective**: $\max_\pi \mathbb{E}[r(x,y) - \beta \text{KL}(\pi \| \pi^{\text{SFT}})]$

**DPO Loss**: $\mathcal{L} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi(y_w|x)}{\pi^{\text{ref}}(y_w|x)} - \beta \log \frac{\pi(y_l|x)}{\pi^{\text{ref}}(y_l|x)}\right)\right]$

### Real-World Impact

- **ChatGPT**: Trained with RLHF for helpfulness
- **Claude**: Constitutional AI for safety
- **Sparrow (DeepMind)**: RLHF for dialogue safety
- **Code generation**: GitHub Copilot uses RLHF

### Next Steps

**Lesson 16b**: Implement RLHF pipeline with:
- Reward model training from preferences
- PPO-RLHF implementation
- DPO implementation
- Comparison experiments

## Further Reading

**Foundational Papers:**
1. Christiano et al. (2017) - "Deep Reinforcement Learning from Human Preferences"
2. Ziegler et al. (2019) - "Fine-Tuning Language Models from Human Preferences"
3. Ouyang et al. (2022) - "Training language models to follow instructions with human feedback" (InstructGPT)
4. Rafailov et al. (2023) - "Direct Preference Optimization"
5. Bai et al. (2022) - "Constitutional AI: Harmlessness from AI Feedback"

**Advanced Topics:**
- Anthropic's RLHF papers on Constitutional AI
- OpenAI's alignment research
- DeepMind's Sparrow dialogue agent
- Recursive Reward Modeling (Leike et al.)

**Practical Resources:**
- Hugging Face TRL (Transformer Reinforcement Learning) library
- OpenAI's alignment forum
- Anthropic's research blog